# OCTA MAE — Phase 2 Pretraining

Cross-modal do-pretraining na **párových** SVP + DCP snímkach (`has_dcp == 1`).
Načíta encoder z Phase 1 — ak súbor neexistuje, tréning začne s random váhami.


## Bunka 1 — Imports + Cesty

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "scripts")
from octa_mae_core import *

BASE_DIR = Path().resolve()

EXCEL_PATH = BASE_DIR.parent / "data" / "master_excels" / "master_table.xlsx"
DATA_ROOT  = BASE_DIR.parent / "data"

# ── Phase 1 encodery ──────────────────────────────────────────────────────────
ENCODER_BASELINE = BASE_DIR / "results" / "phase1" / "baseline" / "pretrained" / "final_encoder.pth"
ENCODER_BALANCED = BASE_DIR / "results" / "phase1" / "balanced" / "pretrained" / "final_encoder.pth"

for name, path in [("Baseline", ENCODER_BASELINE), ("Balanced", ENCODER_BALANCED)]:
    if path and Path(path).exists():
        print(f"Phase1 {name} encoder nájdený : {path}")
    else:
        print(f"Phase1 {name} encoder NENÁJDENÝ — Phase 2 začne s random váhami!")
        print(f"   Hľadaná cesta: {path}")

print()
print(f"Device : {device}")
if torch.cuda.is_available():
    print(f"GPU    : {torch.cuda.get_device_name(0)}")
    free, total = torch.cuda.mem_get_info()
    print(f"VRAM   : {free/1024**3:.1f} GB free / {total/1024**3:.1f} GB total")
print(f"wandb  : {'available' if WANDB_AVAILABLE else 'not installed — logging disabled'}")

---
## Bunka 2 — Phase 2 Baseline
MultiMAE bez GRL. Načíta encoder z **Phase 1 Baseline**.

In [ ]:
p2_baseline_cfg = Phase2Config(
    excel_path          = EXCEL_PATH,
    data_root           = DATA_ROOT,
    output_base         = str(BASE_DIR / "results" / "phase2" / "baseline"),
    phase1_encoder_path = ENCODER_BASELINE,   

    # Tréning
    epochs        = 150,
    batch_size    = 32,
    lr            = 1.5e-4,
    warmup_epochs = 20,
    save_every    = 25,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 2 Baseline...")
p2_baseline_encoder, p2_baseline_path = train_phase2(p2_baseline_cfg)

print(f"\nPhase 2 Baseline hotovo!")
print(f"   Encoder uložený: {p2_baseline_path}")

---
## Bunka 3 — Phase 2 Balanced
MultiMAE bez GRL. Načíta encoder z **Phase 1 Balanced**.

In [ ]:
p2_balanced_cfg = Phase2Config(
    excel_path          = EXCEL_PATH,
    data_root           = DATA_ROOT,
    output_base         = str(BASE_DIR / "results" / "phase2" / "balanced"),
    phase1_encoder_path = ENCODER_BALANCED,   

    # Tréning
    epochs        = 150,
    batch_size    = 32,
    lr            = 1.5e-4,
    warmup_epochs = 20,
    save_every    = 25,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 2 Balanced...")
p2_balanced_encoder, p2_balanced_path = train_phase2(p2_balanced_cfg)

print(f"\nPhase 2 Balanced hotovo!")
print(f"   Encoder uložený: {p2_balanced_path}")

---
## Phase 2 + GRL
MultiMAE + Gradient Reversal Layer (domain adversarial). Načíta encoder z **Phase 1 Balanced**.

In [ ]:
p2_grl_cfg = Phase2Config(
    excel_path          = EXCEL_PATH,
    data_root           = DATA_ROOT,
    output_base         = str(BASE_DIR / "results" / "phase2" / "grl"),
    phase1_encoder_path = ENCODER_BALANCED,   

    # Tréning
    epochs        = 150,
    batch_size    = 32,
    lr            = 1.5e-4,
    warmup_epochs = 20,
    save_every    = 25,

    # GRL parametre
    domain_loss_weight = 0.10,
    grl_max_lambda     = 0.10,
    domain_hidden_dim  = 256,
    domain_dropout     = 0.20,

    wandb_project  = "octa-mae",
    wandb_entity   = None,
    wandb_run_name = None,
)

print("Spúšťam Phase 2 + GRL...")
p2_grl_encoder, p2_grl_path = train_phase2_grl(p2_grl_cfg)

print(f"\nPhase 2 + GRL hotovo!")
print(f"   Encoder uložený: {p2_grl_path}")